In [2]:
# In a notebook from repo root:
from pathlib import Path
from transformers import AutoFeatureExtractor
from dann.config_utils_dann import load_yaml
from dann.dataset_utils_dann import prepare_encoded_datasets_dann
from dann.trainer_utils_dann import AudioDataCollatorWithSpeaker

cfg = load_yaml(Path("task2_dann_mms300m.yaml"))
feat = AutoFeatureExtractor.from_pretrained(cfg["model"]["id"], do_normalize=True, return_attention_mask=True)
prep = prepare_encoded_datasets_dann(cfg, feat, seed=cfg.get("seed", 42))
collator = AudioDataCollatorWithSpeaker(feat, prep.model_input_name)

batch = collator([prep.train_dataset[i] for i in range(8)])
assert "labels" in batch and "speaker_labels" in batch
assert prep.model_input_name in batch

Train columns: ['audio_filepath', 'duration', 'speaker_id', 'scenario', 'age_group', 'job_type', 'area', 'language']
Eval columns: ['audio_filepath', 'duration', 'speaker_id', 'scenario', 'age_group', 'job_type', 'area', 'language']
Dataset badrex/nnti-dataset-full has 8689 train samples.
Dataset badrex/nnti-dataset-full has 3300 eval samples.


Map: 100%|██████████| 3300/3300 [00:05<00:00, 615.78 examples/s]


In [3]:
# Forward shape check:
from dann.model_utils_dann import build_dann_audio_classification_model

model = build_dann_audio_classification_model(
    config=cfg,
    label2id=prep.label2id,
    id2label=prep.id2label,
    num_speakers=len(prep.speaker2id),
)
out = model(**{k: v for k, v in batch.items() if k in [prep.model_input_name, "attention_mask"]})
print(out.logits.shape, out.speaker_logits.shape)

/Users/behkamfallah/Documents/GitHub/indic-language-identification/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/mms-300m and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


torch.Size([8, 22]) torch.Size([8, 111])
